In [ ]:

!pip install faiss-cpu llama-index python-dotenv

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceWindowNodeParser, SentenceSplitter
from llama_index.core import VectorStoreIndex
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
import faiss
import os
import sys
from dotenv import load_dotenv
from pprint import pprint


load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')


EMBED_DIMENSION=512
Settings.llm = OpenAI(model="gpt-3.5-turbo")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=EMBED_DIMENSION)

In [ ]:

import os
os.makedirs('data', exist_ok=True)


!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/sourabhvamdevan/langchain-projects/main/data/Understanding_Climate_Change.pdf


In [ ]:
path = "data/"
reader = SimpleDirectoryReader(input_dir=path, required_exts=['.pdf'])
documents = reader.load_data()
print(documents[0])

In [ ]:

fais_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=fais_index)

## Ingestion Pipelines

### Ingestion Pipeline with Sentence Splitter

In [ ]:
base_pipeline = IngestionPipeline(
    transformations=[SentenceSplitter()],
    vector_store=vector_store
)

base_nodes = base_pipeline.run(documents=documents)

### Ingestion Pipeline with Sentence Window

In [ ]:
node_parser = SentenceWindowNodeParser(
 
    window_size=3,
   
    window_metadata_key="window",
  
    original_text_metadata_key="original_sentence"
)


pipeline = IngestionPipeline(
    transformations=[node_parser],
    vector_store=vector_store,
)

windowed_nodes = pipeline.run(documents=documents)

In [ ]:
query = "Explain the role of deforestation and fossil fuels in climate change"

In [ ]:

base_index = VectorStoreIndex(base_nodes)

base_query_engine = base_index.as_query_engine(
    similarity_top_k=1,
)

base_response = base_query_engine.query(query)

print(base_response)

#### Print Metadata of the Retrieved Node

In [ ]:
pprint(base_response.source_nodes[0].node.metadata)

In [ ]:

windowed_index = VectorStoreIndex(windowed_nodes)


windowed_query_engine = windowed_index.as_query_engine(
    similarity_top_k=1,
    node_postprocessors=[
        MetadataReplacementPostProcessor(
            target_metadata_key="window" 
            )
        ],
)


windowed_response = windowed_query_engine.query(query)

print(windowed_response)

#### Print Metadata of the Retrieved Node

In [ ]:

pprint(windowed_response.source_nodes[0].node.metadata)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--context-enrichment-window-around-chunk-with-llamaindex)